# Task
Afinar el modelo NousResearch/Llama-2-7b-chat-hf usando la biblioteca `transformers` con un conjunto de datos cargado desde un archivo JSON.

## Instalar bibliotecas necesarias

### Subtask:
Instalar las bibliotecas `transformers`, `datasets`, `peft`, `trl`, y `accelerate` para la afinación.


**Reasoning**:
Install the necessary libraries for model fine-tuning using pip.



In [1]:
%pip install transformers==4.41.2 accelerate==0.30.0 datasets peft==0.11.1 trl bitsandbytes==0.42.0 triton==2.2.0

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.8/43.8 kB 2.1 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of trl to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of trl to determine which version is compatible with other requirements. This could take a while.
INFO: This is taking longer than usual. You might need to provide the dependency resolver with stricter constraints to reduce runtime. See https://pip.pypa.io/warnings/backtracking for guidance. If you want to abort this run, press Ctrl + C.
INFO: pip is looking at multiple versions of torch to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of torch to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 51.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━

## Cargar el modelo y el tokenizador

### Subtask:
Cargar el modelo preentrenado NousResearch/Llama-2-7b-chat-hf y su tokenizador asociado.


**Reasoning**:
Import the necessary classes from the transformers library and define the model name.



In [2]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch

model_name = "NousResearch/Llama-2-7b-chat-hf"

**Reasoning**:
Load the tokenizer and the model using the defined model name, specifying the torch_dtype for potential GPU usage.



In [3]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",        # nf4 = NormalFloat4, recomendado
    bnb_4bit_use_double_quant=True,   # doble quantization para más compresión
    bnb_4bit_compute_dtype="float16"  # cálculos en FP16
)

In [4]:
tokenizer = AutoTokenizer.from_pretrained(model_name)
base_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16,
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/746 [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/21.0 [00:00<?, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/9.98G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/3.50G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/200 [00:00<?, ?B/s]

## Cargar y preparar el conjunto de datos

### Subtask:
Cargar el conjunto de datos desde el archivo JSON y prepararlo en el formato adecuado para el entrenamiento. Esto puede implicar formatear los datos de texto para que se ajusten al formato de conversación esperado por el modelo.


**Reasoning**:
Import the necessary function to load the dataset and define the formatting function.



**Reasoning**:
Load the dataset from the JSON file and apply the formatting function to create a new column.



In [13]:
import json
from datasets import load_dataset, Dataset
from sklearn.model_selection import train_test_split
import pandas as pd

def format_example(example):
    # Assuming 'tools' is a list within the 'response' dictionary
    tools = example.get("response", {}).get("tools")

    # Create a string representation of the tools, handling None
    tools_str = str(sorted(tools)) if isinstance(tools, list) and tools else "null"

    # Combine input and response (justification in this case) into a single text field
    # You might adjust this based on how you want to format the conversation for fine-tuning
    formatted_text = f"<s>[INST] {example['input']} [/INST] {json.dumps(example['response'], ensure_ascii=False)}</s>"

    return {"formatted_text": formatted_text, "tools_label": tools_str}

# Load the dataset from the JSON file
# Ensure 'dataset.json' exists and has the correct structure
try:
    with open("dataset.json", "r") as f:
        data = json.load(f)
except FileNotFoundError:
    print("Error: dataset.json not found. Please upload the dataset file.")
    data = [] # Initialize with empty list to avoid further errors


In [14]:
# Convert the list of dictionaries to a Dataset
dataset = Dataset.from_list(data)

# Apply the formatting function and create the 'tools_label' for stratification
dataset = dataset.map(format_example)

# Convert to pandas DataFrame for stratified splitting
df = dataset.to_pandas()

# Perform stratified split based on the 'tools_label'
if not df.empty:
    train_df, test_df = train_test_split(
        df,
        test_size=0.2,  # 20% for testing
        stratify=df['tools_label'],
        random_state=42 # for reproducibility
    )

    # Convert back to Dataset objects
    train_dataset = Dataset.from_pandas(train_df)
    test_dataset = Dataset.from_pandas(test_df)

    # Now you have train_dataset and test_dataset ready for fine-tuning and evaluation
    print("Dataset loaded and split successfully.")
    print("Train dataset size:", len(train_dataset))
    print("Test dataset size:", len(test_dataset))
    print("Tools label distribution in train dataset:")
    print(train_dataset.to_pandas()['tools_label'].value_counts(normalize=True))
    print("Tools label distribution in test dataset:")
    print(test_dataset.to_pandas()['tools_label'].value_counts(normalize=True))

else:
    print("Dataset is empty. Cannot perform split.")
    train_dataset = Dataset.from_list([])
    test_dataset = Dataset.from_list([])

Map:   0%|          | 0/1082 [00:00<?, ? examples/s]

Dataset loaded and split successfully.
Train dataset size: 865
Test dataset size: 217
Tools label distribution in train dataset:
tools_label
['object_detection']                      0.268208
['describe_scene']                        0.203468
['ocr']                                   0.198844
[None]                                    0.152601
['describe_scene', 'ocr']                 0.073988
['object_detection', 'ocr']               0.055491
['describe_scene', 'object_detection']    0.047399
Name: proportion, dtype: float64
Tools label distribution in test dataset:
tools_label
['object_detection']                      0.267281
['describe_scene']                        0.202765
['ocr']                                   0.198157
[None]                                    0.156682
['describe_scene', 'ocr']                 0.073733
['object_detection', 'ocr']               0.055300
['describe_scene', 'object_detection']    0.046083
Name: proportion, dtype: float64


In [15]:
def tokenize(batch):
    tokens = tokenizer(batch["formatted_text"], padding="max_length", truncation=True, max_length=512)
    tokens["labels"] = tokens["input_ids"].copy()
    return tokens

train_dataset_tk = train_dataset.map(tokenize, batched=True)
test_dataset_tk = test_dataset.map(tokenize, batched=True)


Map:   0%|          | 0/865 [00:00<?, ? examples/s]

Map:   0%|          | 0/217 [00:00<?, ? examples/s]

## Configurar el entrenamiento

### Subtask:
Definir los parámetros de entrenamiento, como la tasa de aprendizaje, el número de épocas, el tamaño del lote, etc.


**Reasoning**:
Import the `TrainingArguments` class and instantiate it with the required parameters for training.



In [16]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./llama_finetuned",
    evaluation_strategy="steps",
    eval_steps=50,
    logging_steps=10,
    save_steps=200,
    save_total_limit=2,
    learning_rate=2e-4,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    num_train_epochs=3,
    gradient_accumulation_steps=8,
    report_to="none",
    fp16=False,
    bf16=True
)


/usr/local/lib/python3.12/dist-packages/transformers/training_args.py:1474: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


## Configurar la afinación (fine-tuning) con peft/lora

### Subtask:
Configurar la estrategia de afinación eficiente (PEFT), como LoRA, para reducir los recursos computacionales necesarios.


**Reasoning**:
Import the necessary classes from the `peft` library and define the LoRA configuration.



In [20]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)


**Reasoning**:
Wrap the base model with the defined LoRA configuration using `get_peft_model`.



In [23]:
from peft import get_peft_model

finetune_model = get_peft_model(base_model, lora_config)
finetune_model.print_trainable_parameters()

OutOfMemoryError: CUDA out of memory. Tried to allocate 2.00 MiB. GPU 

## Entrenar el modelo

### Subtask:
Iniciar el proceso de entrenamiento utilizando la biblioteca `trl` (Transformer Reinforcement Learning) u otra herramienta de entrenamiento adecuada.


**Reasoning**:
Import the `SFTTrainer` class and instantiate it with the required parameters to prepare for training.



In [19]:
from trl import SFTTrainer
trainer = SFTTrainer(
     model=finetune_model,
     train_dataset=train_dataset_tk,
     eval_dataset=test_dataset_tk,
     peft_config=lora_config,
     dataset_text_field="formatted_text",
     max_seq_length=512,
     tokenizer=tokenizer,
     args=training_args
     )
trainer.train()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_deprecation.py:100: FutureWarning: Deprecated argument(s) used in '__init__': dataset_text_field, max_seq_length. Will not be supported from version '1.0.0'.

Deprecated positional argument(s) used in SFTTrainer, please use the SFTConfig to set these arguments instead.
  warnings.warn(message, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/training_args.py:1474: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/trl/trainer/sft_trainer.py:283: UserWarning: You passed a `max_seq_length` argument to the SFTTrainer, the value you passed will override the one in the `SFTConfig`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/trl/trainer/sft_trainer.py:321: UserWarning: You passed a `dataset_text_field` argument to the SFTTrainer, the value you passed will over

FP4 quantization state not initialized. Please call .cuda() or .to(device) on the LinearFP4 layer first.


AssertionError: 

In [ ]:
import pandas as pd
from tqdm import tqdm

def generate_response(model, tokenizer, prompt, max_new_tokens=70):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            temperature=0.7,
            top_p=0.9
        )
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

# Tomamos algunos ejemplos de test_dataset
sampled = test_dataset.shuffle(seed=42).select(range(20))  # coge 20 ejemplos
results = []

for example in tqdm(sampled):
    inp = example["input"]

    # salida modelo original
    out_base = generate_response(base_model, tokenizer, inp)

    # salida modelo fine-tuneado
    out_ft = generate_response(finetune_model, tokenizer, inp)

    results.append({
        "input": inp,
        "output_original": out_base,
        "output_finetuned": out_ft
    })

# Guardar en DataFrame
df_results = pd.DataFrame(results)

# Exportar a CSV y JSON
df_results.to_csv("comparison_outputs.csv", index=False)
df_results.to_json("comparison_outputs.json", orient="records", force_ascii=False, indent=2)

print("Comparación guardada en CSV y JSON.")


In [ ]:
sampled[0]

In [ ]:
 with torch.no_grad():
        outputs = model.generate(
            tokenizer(sampled[0]['input'], return_tensors="pt").to(model.device),
            max_new_tokens=70,
            do_sample=False,
            temperature=0.7,
            top_p=0.9
        )

In [ ]:
# Guardar solo los adaptadores
finetune_model.save_pretrained("llama_finetuned_lora")
tokenizer.save_pretrained("llama_finetuned_lora")


In [ ]:
# from peft import PeftModel

# # cargar modelo base
# base_model = AutoModelForCausalLM.from_pretrained(
#     model_name,
#     torch_dtype=torch.float16,
#     device_map="auto"
# )

# # aplicar adaptadores LoRA
# model = PeftModel.from_pretrained(base_model, "llama_finetuned_lora")
# tokenizer = AutoTokenizer.from_pretrained("llama_finetuned_lora")